In [ ]:
!pip install duckdb
!pip install pandas

In [1]:
import os
os.chdir("../")

### Import all tables

In [2]:
import duckdb

con = duckdb.connect()

# Charger title.basics.tsv
con.execute("""
    CREATE TABLE title_basics AS
    SELECT *
    FROM read_csv(
        'data/raw/title.basics.tsv',
        delim='\t',
        header=true,
        nullstr='\\N',
        columns={
            'tconst': 'VARCHAR',
            'titleType': 'VARCHAR',
            'primaryTitle': 'VARCHAR',
            'originalTitle': 'VARCHAR',
            'isAdult': 'BOOLEAN',
            'startYear': 'INTEGER',
            'endYear': 'INTEGER',
            'runtimeMinutes': 'INTEGER',
            'genres': 'VARCHAR'
        }
    )
""")

# Intéressant : ordering est lié à l'importance de l'acteur dans le film (rôle principal si ordering <=5 par exemple)
con.execute("""
    CREATE TABLE title_principals AS
    SELECT *
    FROM read_csv(
        'data/raw/title.principals.tsv',
        delim='\t',
        header=true,
        nullstr='\\N',
        columns={
            'tconst': 'VARCHAR',
            'ordering': 'INTEGER', 
            'nconst': 'VARCHAR',
            'category': 'VARCHAR',
            'job': 'VARCHAR',
            'characters': 'VARCHAR'
        }
    )
""")

con.execute("""
    CREATE TABLE name_basics AS
    SELECT *
    FROM read_csv(
        'data/raw/name.basics.tsv',
        delim='\t',
        header=true,
        nullstr='\\N',
        columns={
            'nconst': 'VARCHAR',
            'primaryName': 'VARCHAR', 
            'birthYear': 'INTEGER',
            'deathYear': 'INTEGER',
            'primaryProfession': 'VARCHAR',
            'knownForTitles': 'VARCHAR'
        }
    )
""")

con.execute("""
    CREATE TABLE title_akas AS
    SELECT *
    FROM read_csv(
        'data/raw/title.akas.tsv',
        delim='\t',
        header=true,
        nullstr='\\N',
        columns={
            'titleId':        'VARCHAR',
            'ordering':       'INTEGER',
            'title':          'VARCHAR',
            'region':         'VARCHAR',
            'language':       'VARCHAR',
            'types':          'VARCHAR',
            'attributes':     'VARCHAR',
            'isOriginalTitle':'INTEGER'
        }
    )
""")



In [ ]:
con.execute("SELECT * FROM title_basics LIMIT 5").df()
con.execute("SELECT * FROM title_principals LIMIT 5").df()
con.execute("SELECT * FROM name_basics LIMIT 5").df()
con.execute("SELECT * FROM title_akas LIMIT 5").df()

,titleId,ordering,title,region,language,types,attributes,isOriginalTitle
0,tt0000001,1,Carmencita,None,None,original,None,1
1,tt0000001,2,Carmencita,DE,None,None,literal title,0
2,tt0000001,3,Carmencita,US,None,imdbDisplay,None,0
3,tt0000001,4,Carmencita - spanyol tánc,HU,None,imdbDisplay,None,0
4,tt0000001,5,Καρμενσίτα,GR,None,imdbDisplay,None,0
5,tt0000001,6,Карменсита,RU,None,imdbDisplay,None,0
6,tt0000001,7,Карменсіта,UA,None,imdbDisplay,None,0
7,tt0000001,8,カルメンチータ,JP,ja,imdbDisplay,None,0
8,tt0000002,1,Le clown et ses chiens,None,None,original,None,1
9,tt0000002,2,A bohóc és kutyái,HU,None,imdbDisplay,None,0


### Create joined table

In [8]:
nb_of_years = 15
min_number_of_movies_by_actor = 3

con.execute(f"""
    CREATE TABLE films_acteurs2 AS
    SELECT
        -- Infos film
        b.tconst,
        b.primaryTitle      AS title,
        b.originalTitle,
        b.startYear,
        b.genres,

        -- Infos région (depuis akas)
        a.region,
        a.language,

        -- Infos acteur
        p.nconst,
        p.ordering,
        p.category,
        n.primaryName,
        n.birthYear,
        n.deathYear

    FROM title_basics b
    JOIN title_principals p  ON b.tconst = p.tconst
    JOIN name_basics n       ON n.nconst = p.nconst

    -- Meilleure approximation de la région de production
    LEFT JOIN (
        SELECT DISTINCT ON (titleId)
            titleId,
            region,
            language
        FROM title_akas
        WHERE region IS NOT NULL
        ORDER BY
            titleId,
            isOriginalTitle DESC,           -- priorité au titre original
            (types = 'original') DESC,      -- puis type 'original'
            ordering ASC                    -- puis premier ordering
    ) a ON a.titleId = b.tconst

    WHERE
        b.titleType = 'movie'
        AND b.startYear >= YEAR(CURRENT_DATE) - {nb_of_years}
        AND b.startYear <= YEAR(CURRENT_DATE)
        AND b.startYear IS NOT NULL
        AND p.category IN ('actor', 'actress')
        AND p.nconst IN (
            SELECT p2.nconst
            FROM title_principals p2
            JOIN title_basics b2 ON b2.tconst = p2.tconst
            WHERE
                b2.titleType = 'movie'
                AND b2.startYear >= YEAR(CURRENT_DATE) - {nb_of_years}
                AND b2.startYear <= YEAR(CURRENT_DATE)
                AND b2.startYear IS NOT NULL
                AND p2.category IN ('actor', 'actress')
            GROUP BY p2.nconst
            HAVING COUNT(DISTINCT b2.tconst) > {min_number_of_movies_by_actor}
        )

    ORDER BY b.startYear DESC, b.tconst, p.ordering
""")

print(con.execute("SELECT COUNT(*) FROM films_acteurs").fetchone())


(952696,)


In [5]:
con.execute("SELECT * FROM films_acteurs LIMIT 10").df()

,tconst,title,originalTitle,startYear,genres,region,language,nconst,ordering,category,primaryName,birthYear,deathYear
0,tt0111463,Totschweigen,Totschweigen,2026,"Documentary,Drama",AT,None,nm0092498,2,actress,Kornelia Boje,1942,<NA>
1,tt0360010,Specter's Rock,Specter's Rock,2026,"Comedy,Mystery,Thriller",GB,None,nm0007491,1,actress,Bonnie Aarons,<NA>,<NA>
2,tt0360010,Specter's Rock,Specter's Rock,2026,"Comedy,Mystery,Thriller",GB,None,nm1346092,5,actor,Josh Eisenstadt,<NA>,<NA>
3,tt0427340,Masters of the Universe,Masters of the Universe,2026,"Action,Adventure,Family",AU,None,nm1072555,1,actress,Morena Baccarin,1979,<NA>
4,tt0427340,Masters of the Universe,Masters of the Universe,2026,"Action,Adventure,Family",AU,None,nm0252961,2,actor,Idris Elba,1972,<NA>
5,tt0427340,Masters of the Universe,Masters of the Universe,2026,"Action,Adventure,Family",AU,None,nm0252961,3,actor,Idris Elba,1972,<NA>
6,tt0427340,Masters of the Universe,Masters of the Universe,2026,"Action,Adventure,Family",AU,None,nm1325419,4,actress,Kristen Wiig,1973,<NA>
7,tt0427340,Masters of the Universe,Masters of the Universe,2026,"Action,Adventure,Family",AU,None,nm5954280,5,actor,Nicholas Galitzine,1994,<NA>
8,tt0427340,Masters of the Universe,Masters of the Universe,2026,"Action,Adventure,Family",AU,None,nm5954280,6,actor,Nicholas Galitzine,1994,<NA>
9,tt0427340,Masters of the Universe,Masters of the Universe,2026,"Action,Adventure,Family",AU,None,nm1555340,7,actress,Alison Brie,1982,<NA>


In [9]:
df = con.execute("SELECT * FROM films_acteurs2").df()
df.to_csv(f'data/processed/films_acteurs_period:{nb_of_years}_minfilm{min_number_of_movies_by_actor}.csv', index=False)


In [7]:
len(df['nconst'].unique())

203657